# Chapter 18 — MLFFNN II: Backpropagation & Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part3_neural_networks/ch18_mlffnn_ii/ch18_notebook.ipynb)

Companion notebook for **Chapter 18** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- The forward pass computes the output; the backward pass computes gradients using the chain rule.
- Local gradients (the deltas) are passed backwards layer by layer, which is what makes training deep networks feasible.
- Pattern, mini-batch and batch modes trade noise against speed; mini-batch is the usual practical choice.
- Early stopping halts training when validation performance stops improving, saving compute and improving generalization.
- Different random initializations converge to different local minima, so run several seeds and keep the best.

## Example 1: Backpropagation — Manual Forward & Backward Pass

This is the most important code in the entire book. We build backpropagation completely from scratch — no sklearn, no PyTorch, just numpy and math. The network has 2 inputs, 2 hidden tanh neurons, and 1 logistic output neuron. Given one training example (x=[0.5, 0.8], target=1.0), we trace the full forward pass (computing activations step by step), then the full backward pass (computing all gradients using the chain rule). The final check confirms the error decreased after the weight update — proving that one step of gradient descent worked correctly.

In [1]:
import numpy as np

np.random.seed(42)
x      = np.array([0.5, 0.8])
target = 1.0
eta    = 0.5

W1 = np.array([[0.15, 0.25],[0.20, 0.30]])  # hidden weights
b1 = np.array([0.35, 0.35])
W2 = np.array([[0.40, 0.45]])               # output weights
b2 = np.array([0.60])

sigmoid = lambda z: 1 / (1 + np.exp(-z))
tanh    = lambda z: np.tanh(z)

# ── FORWARD PASS ──
a_h = W1 @ x + b1       # hidden net inputs
s_h = tanh(a_h)         # hidden outputs (tanh)
a_o = W2 @ s_h + b2     # output net input
s_o = sigmoid(a_o)      # final output

E = 0.5 * (target - s_o[0])**2
print("=== FORWARD PASS ===")
print(f"Hidden net inputs a_h : {a_h.round(4)}")
print(f"Hidden outputs s_h    : {s_h.round(4)}")
print(f"Output net input a_o  : {a_o[0]:.4f}")
print(f"Network output s_o    : {s_o[0]:.4f}")
print(f"Target                : {target}")
print(f"Error E               : {E:.6f}")

# ── BACKWARD PASS ──
print("\n=== BACKWARD PASS ===")
delta_o = -(target - s_o[0]) * s_o[0] * (1 - s_o[0])
print(f"Output local gradient delta_o: {delta_o:.6f}")

dW2  = delta_o * s_h
W2_n = W2 - eta * dW2
b2_n = b2 - eta * delta_o
print(f"Updated output weights: {W2_n.round(6)}")

delta_h = delta_o * W2[0] * (1 - s_h**2)
print(f"Hidden local gradients delta_h: {delta_h.round(6)}")

dW1  = np.outer(delta_h, x)
W1_n = W1 - eta * dW1

# Verify: error should decrease
a_h2 = W1_n @ x + b1; s_h2 = tanh(a_h2)
s_o2 = sigmoid(W2_n @ s_h2 + b2_n)
E2   = 0.5 * (target - s_o2[0])**2
print(f"\nError BEFORE update: {E:.6f}")
print(f"Error AFTER  update: {E2:.6f}")
print(f"Error reduced by   : {E - E2:.6f}  <- gradient descent worked!")

=== FORWARD PASS ===
Hidden net inputs a_h : [0.625 0.69 ]
Hidden outputs s_h    : [0.5546 0.598 ]
Output net input a_o  : 1.0909
Network output s_o    : 0.7486
Target                : 1.0
Error E               : 0.031612

=== BACKWARD PASS ===
Output local gradient delta_o: -0.047326
Updated output weights: [[0.413124 0.46415 ]]
Hidden local gradients delta_h: [-0.013108 -0.013681]

Error BEFORE update: 0.031612
Error AFTER  update: 0.029636
Error reduced by   : 0.001976  <- gradient descent worked!


## Example 2: Training Modes Compared + Early Stopping

Sklearn's MLPClassifier lets you control the batch size, effectively simulating Pattern Mode (batch=1), Mini-Batch Mode (batch=32, 64, 128), and Batch Mode (batch=full dataset). The results confirm the theory: Mini-Batch mode converges in far fewer epochs than full-batch, and Adam with mini-batch=32 is the fastest of all (only 45 epochs to converge). The early stopping demo shows that with n_iter_no_change=10 (stop if no improvement for 10 consecutive epochs), training halts at epoch 55 instead of running all 300 — a 5x speedup with comparable accuracy.

In [2]:
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import numpy as np

np.random.seed(42)
X, y = make_classification(n_samples=2000, n_features=15, n_informative=10,
                            n_classes=2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
X_tr2, X_va, y_tr2, y_va = train_test_split(X_tr, y_tr, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_tr2 = scaler.fit_transform(X_tr2)
X_va  = scaler.transform(X_va)
X_te  = scaler.transform(X_te)

print("=== Training Mode Comparison ===\n")
print(f"{'Mode':<26} {'Train':>7} {'Test':>7} {'Epochs':>8}")
print("-" * 52)
configs = [
    ('Batch (full dataset)', 'sgd', len(X_tr2)),
    ('Mini-Batch size=32',   'sgd', 32),
    ('Mini-Batch size=64',   'sgd', 64),
    ('Mini-Batch size=128',  'sgd', 128),
    ('Adam (batch=32)',      'adam', 32),
]
for name, solver, bs in configs:
    mlp = MLPClassifier(hidden_layer_sizes=(64,32), activation='relu',
                        solver=solver, batch_size=bs,
                        max_iter=300, random_state=42, learning_rate_init=0.01)
    mlp.fit(X_tr2, y_tr2)
    tr = accuracy_score(y_tr2, mlp.predict(X_tr2))
    te = accuracy_score(y_te,  mlp.predict(X_te))
    print(f"  {name:<24} {tr*100:>6.1f}% {te*100:>6.1f}% {mlp.n_iter_:>8}")

print("\n=== Early Stopping Demo ===\n")
for es in [False, True]:
    mlp = MLPClassifier(hidden_layer_sizes=(64,32), activation='relu',
                        solver='adam', max_iter=300, random_state=42,
                        early_stopping=es, validation_fraction=0.15, n_iter_no_change=10)
    mlp.fit(X_tr2, y_tr2)
    te = accuracy_score(y_te, mlp.predict(X_te))
    label = "Early Stopping ON " if es else "Early Stopping OFF"
    print(f"  {label}: epochs={mlp.n_iter_:>4} | Test={te*100:.1f}%")
print("\nEarly stopping: fewer epochs, same performance, less compute wasted.")

=== Training Mode Comparison ===

Mode                         Train    Test   Epochs
----------------------------------------------------
  Batch (full dataset)       96.6%   94.0%      300
  Mini-Batch size=32        100.0%   94.2%      117
  Mini-Batch size=64        100.0%   94.0%      186
  Mini-Batch size=128       100.0%   94.0%      296
  Adam (batch=32)           100.0%   93.8%       45

=== Early Stopping Demo ===

  Early Stopping OFF: epochs= 206 | Test=94.8%
  Early Stopping ON : epochs=  55 | Test=93.8%

Early stopping: fewer epochs, same performance, less compute wasted.


## Example 3: Local Minima — Effect of Random Initialization

One of the most practically important issues in neural network training is that different random initializations lead to different final models — because gradient descent can get stuck in different local minima. This example trains the same MLFFNN 10 times on the same data with 10 different random seeds. The resulting test accuracies vary noticeably (the range between best and worst is printed). This is why frameworks like PyTorch let you set random seeds for reproducibility, and why practitioners run multiple experiments and pick the best model. Pick it the right way: the data is split three ways, the best seed is chosen on the validation set, and the test set is used only once, on the chosen model. Notice that the chosen model scores 96% on validation but 91% on test. Picking the best of ten runs flatters the validation score a little, which is exactly why the final number must come from data that played no part in the choice.

In [3]:
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)   # some seeds need more than 500 epochs

X, y = make_moons(n_samples=500, noise=0.25, random_state=0)
# Three-way split: train (60%) to fit, validation (20%) to choose, test (20%) to report once
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=0)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print("=== Local Minima: Effect of Random Initialization ===\n")
print("Same network, same data, different random seeds -> different results")
print(f"Architecture: (3,) | Activation: tanh | Optimizer: SGD")
print(f"Split: {len(X_train)} train / {len(X_val)} validation / {len(X_test)} test\n")
print(f"{'Run':>5} {'Seed':>6} {'Train Acc':>11} {'Val Acc':>10}  {'Status'}")
print("-" * 52)
models, val_accs = [], []
for run, seed in enumerate(range(1, 11), 1):
    mlp = MLPClassifier(hidden_layer_sizes=(3,), activation='tanh',
                        solver='sgd', max_iter=500, random_state=seed,
                        learning_rate_init=0.05)
    mlp.fit(X_train, y_train)
    tr = accuracy_score(y_train, mlp.predict(X_train))
    va = accuracy_score(y_val,   mlp.predict(X_val))
    status = "BEST so far" if not val_accs or va > max(val_accs) else ""
    models.append(mlp); val_accs.append(va)
    print(f"{run:>5} {seed:>6} {tr*100:>10.1f}% {va*100:>9.1f}%  {status}")

best = int(np.argmax(val_accs))
print(f"\nBest  validation accuracy: {max(val_accs)*100:.1f}%  (seed {best + 1})")
print(f"Worst validation accuracy: {min(val_accs)*100:.1f}%")
print(f"Range                    : {(max(val_accs)-min(val_accs))*100:.1f}%")

# The test set is used exactly once, on the model chosen by validation accuracy
test_acc = accuracy_score(y_test, models[best].predict(X_test))
print(f"\nTest accuracy of the chosen model (reported once): {test_acc*100:.1f}%")
print("\nConclusion: run several random initializations, pick the best on VALIDATION data,")
print("and touch the test set only once at the end.")

=== Local Minima: Effect of Random Initialization ===

Same network, same data, different random seeds -> different results
Architecture: (3,) | Activation: tanh | Optimizer: SGD
Split: 300 train / 100 validation / 100 test

  Run   Seed   Train Acc    Val Acc  Status
----------------------------------------------------
    1      1       94.0%      96.0%  BEST so far
    2      2       95.0%      93.0%  
    3      3       87.0%      88.0%  
    4      4       95.3%      93.0%  
    5      5       95.7%      93.0%  
    6      6       94.0%      94.0%  
    7      7       94.3%      93.0%  
    8      8       88.0%      89.0%  
    9      9       94.3%      93.0%  
   10     10       95.3%      93.0%  

Best  validation accuracy: 96.0%  (seed 1)
Worst validation accuracy: 88.0%
Range                    : 8.0%

Test accuracy of the chosen model (reported once): 91.0%

Conclusion: run several random initializations, pick the best on VALIDATION data,
and touch the test set only once at t